# Original GuitarSet test: listen and inspect

This notebook inspects the original frozen GuitarSet test comparison. The current progress report focuses on later development experiments and the fast-tempo IDMT comparison. Change `recording_id` to inspect another processed recording. The saved CSVs contain start time, end time, and chord label.

Two selected cases are `05_BN3-119-G_comp` (largest adaptive gain over the long-window baseline) and `04_Funk3-112-C#_comp` (largest decrease). These decoded examples were selected after evaluation from 57 recordings with scoreable intervals; their reference coverage is 37.51% and 87.50%, respectively. The aggregate summary includes all 59 test recordings.


In [ ]:
from pathlib import Path
import csv, html, json
from IPython.display import Audio, Image, HTML, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "chordrec").is_dir())
run = root / "results/test_frozen"
records = json.loads((run / "selection.json").read_text())
recording_id = "00_BN3-119-G_comp"
record = next(r for r in records if r["id"] == recording_id)
print(recording_id)
display(Audio(filename=str(root / record["audio"])))
figure = run / "examples" / recording_id / "timeline.png"
if figure.exists():
    display(Image(filename=str(figure)))
else:
    import sys
    if str(root) not in sys.path: sys.path.insert(0, str(root))
    from chordrec.visualize import make_example
    make_example(run, record, json.loads((run / "config.json").read_text()), root)
    display(Image(filename=str(figure)))
print("Interactive playback:", figure.with_name("playback.html"))

## Saved predictions

The complete files are under `results/test_frozen/predictions/<recording_id>/`. `X` in the figure marks unsupported reference labels, excluded from scoring. Major/minor reductions follow `mir_eval.majmin`. The selected fixed short-window weight is zero, so its predictions match the long-window baseline.

In [ ]:
path = run / "predictions" / recording_id / "adaptive_viterbi.csv"
with path.open() as stream:
    rows = list(csv.DictReader(stream))
print(f"{len(rows)} predicted intervals; showing first 15. CSV: {path}")
body = "".join("<tr>" + "".join(f"<td>{html.escape(str(r[k]))}</td>" for k in ("start", "end", "label")) + "</tr>" for r in rows[:15])
display(HTML("<table><tr><th>Start (s)</th><th>End (s)</th><th>Chord</th></tr>" + body + "</table>"))

In [ ]:
summary = json.loads((run / "summary.json").read_text())
for row in summary:
    if row["decoder"] == "viterbi":
        print(f"{row['method']:8s}  duration accuracy {row['duration_accuracy']:.2%}  boundary F1 {row['boundary_f1']:.2%}")
print("Coverage is the fraction of recording time with supported reference chords. Test settings were fixed using development data.")